# 멀티에이전트 오케스트레이터 — 수직 슬라이스 PoC (v0.3, 실 MCP)
**주제:** 건설자재 객체탐지 모델 후보 3종(Faster R-CNN / YOLOv8 / RT-DETR) 비교·분석.

구조 = **ORC(순수 라우터) + 5노드(PLAN/OFFER/TOOL/MEMORY/Verify)** 상태기계. 노드들은 단 하나의 딕셔너리(`AgentContext`)를 돌려가며 고친다.

**v0.3의 핵심 변화 — mock → 실 MCP.** TOOL이 더 이상 손으로 적은 가짜 숫자를 쓰지 않는다. 별도 프로세스로 뜨는 **웹검색 MCP 서버(`mcp_model_tool.py`, 키리스 DuckDuckGo)**를 `langchain-mcp-adapters`로 호출해 **외부 실제 출처(real URL)**에서 데이터를 가져온다. → "데이터를 내가/LLM이 지어낸 것 아니냐"는 의심을 구조적으로 해소.

## 실행 방법
- **실 LLM(기본):** `.env`의 `CLAUDE_KEY` 자동 로드 → OFFER 분석/요약·s4 judge가 실제 추론.
- **실 웹검색(기본):** MCP 서버가 DuckDuckGo로 라이브 검색.
- **오프라인 검증(네트워크 없는 곳):** 첫 셀에서 `SLICE_OFFLINE=1`(LLM 스텁) + `MCP_OFFLINE=1`(검색 고정 스니펫) 주석 해제.

> 두 시나리오: **A** happy path(한 대상 검색실패→재시도 회복→완주), **B** 차단기(영구 검색실패→유계 재시도 소진→FAILED).

In [1]:
# 0. 설치 (최초 1회). 주피터에서 주석 풀고 실행.
# %pip install langgraph langchain langchain-anthropic langgraph-checkpoint-sqlite aiosqlite langchain-mcp-adapters mcp ddgs python-dotenv

import os
# --- API 키 로드: .env에서 읽어 셸 히스토리/화면 노출 없이 주입 (export 불필요) ---
ENV_PATH = "/Users/karla/Documents/constgx/scripts/.env"
try:
    from dotenv import load_dotenv
    load_dotenv(ENV_PATH)              # .env의 CLAUDE_KEY 등을 os.environ에 로드
except ImportError:
    pass

# 오프라인 스위치 (네트워크/키 없는 곳에서만 주석 해제)
# os.environ["SLICE_OFFLINE"] = "1"   # LLM 스텁
# os.environ["MCP_OFFLINE"]   = "1"   # 웹검색 고정 스니펫
OFFLINE = os.environ.get("SLICE_OFFLINE") == "1"
print("SLICE_OFFLINE:", OFFLINE, "| MCP_OFFLINE:", os.environ.get("MCP_OFFLINE")=="1",
      "| CLAUDE_KEY 로드됨:", bool(os.environ.get("CLAUDE_KEY")))

SLICE_OFFLINE: False | MCP_OFFLINE: False | CLAUDE_KEY 로드됨: True


## 과금 구조 복기 가드 (`cost_notice`)
현재는 **LLM 호출당 과금만**(유휴 0). 유료 검색 API·상시 서버·LangSmith 유료플랜으로 넘어가면 과금 구조가 달라진다 → 이 가드가 env를 감지해 그 시점에 경고한다. 상세표는 `COST-MODEL.md`.

In [2]:
def cost_notice():
    notes = []
    if os.environ.get("LANGSMITH_API_KEY") or os.environ.get("LANGCHAIN_TRACING_V2") == "true":
        notes.append("LangSmith 트레이싱 ON → 트레이스 전송량 단위 과금(무료 한도 초과분).")
    for k in ("TAVILY_API_KEY", "SERPER_API_KEY", "BRAVE_API_KEY"):
        if os.environ.get(k):
            notes.append(f"{k} 감지 → 유료 검색 API는 검색 호출당 과금(키리스 DDG와 다른 비용축).")
    if notes:
        print("[COST] 과금 구조 변화 감지 — COST-MODEL.md 복기:")
        for n in notes: print("  -", n)
    else:
        print("[COST] 현재: LLM 호출당 과금만(유휴 0). 유료검색/상시서버/LangSmith 진입 시 COST-MODEL.md 참조.")

cost_notice()

[COST] 현재: LLM 호출당 과금만(유휴 0). 유료검색/상시서버/LangSmith 진입 시 COST-MODEL.md 참조.


## §2. State schema = AgentContext
노드 간에 주고받는 **유일한 딕셔너리**. `history`는 원문이 아니라 digest(요약)만.

In [3]:
from typing import TypedDict, Literal, Optional, Any
from typing_extensions import NotRequired

class PlanStep(TypedDict):
    step_id: str; subgoal: str; priority: int
    depends_on: list[str]; status: Literal["pending","running","done","failed"]

class Constraints(TypedDict):
    definition: dict[str, Any]; allowed_tools: list[str]
    forbidden: list[str]; acceptance: dict[str, Any]

class HistoryEntry(TypedDict):
    t: str; actor: Literal["ORC","PLAN","OFFER","TOOL","MEMORY","Verify"]
    action: str; input_digest: str; result_digest: str
    status: Literal["ok","fail"]; cause: Optional[str]

class Budget(TypedDict):
    iter: int; max_iter: int; retry: int; consec_fail: int; fail_threshold: int

class Verdict(TypedDict):
    passed: bool; cause: Optional[str]; detail: NotRequired[str]

class AgentContext(TypedDict):
    goal: str; constraints: Constraints; plan: list[PlanStep]
    cursor: Optional[str]; artifacts: dict[str, Any]; scratch: dict[str, Any]
    history: list[HistoryEntry]; budget: Budget; verdict: Optional[Verdict]

## LLM 호출 (실 LLM 경로 기본 / OFFLINE 스텁)
`SYS` 시스템 프롬프트가 실 LLM 품질의 핵심.

In [4]:
import json
from datetime import datetime, timezone

SYS = {
 "compare": ("너는 ML 엔지니어다. 주어진 검색 스니펫(specs)을 비교기준(criteria)별로 분석하라. "
             "각 기준마다 어느 모델이 우세한지와 근거(스니펫의 수치)를 한 줄씩. 스니펫에 없는 값은 '자료없음'이라 명시. 추측 금지."),
 "summarize": ("너는 PM이다. 차이분석(analysis)을 바탕으로 용도별 추천(실시간성/정확도/균형)을 작성하라. "
               "주어진 모든 비교기준을 빠짐없이 언급하되, 스니펫에 값이 없는 기준은 '자료없음'이라 명시하라. "
               "각 추천 근거에 출처 URL(specs의 sources)을 괄호로 반드시 표기."),
 "judge": ("너는 품질 검수자다. summary가 (1) 각 criteria를 다뤘는지 — 수치든 '자료없음'이든 *언급*했으면 다룬 것으로 인정 — "
           "(2) 출처(sources) URL이 표기됐는지 점검. 둘 다 충족이면 첫 줄에 정확히 'PASS'만, 아니면 'FAIL: <부족한 점>'만 출력."),
}

def llm_call(purpose: str, payload: dict) -> str:
    if OFFLINE:
        return {"compare":"차이분석(stub): 스니펫 기준 YOLOv8=고FPS, Faster R-CNN=고정확도, RT-DETR=균형.",
                "summarize":"요약(stub): 실시간성=YOLOv8, 정확도=Faster R-CNN, 균형=RT-DETR. (근거: 검색 출처 URL)",
                "judge":"PASS"}.get(purpose, "stub")
    from langchain_anthropic import ChatAnthropic
    from langchain_core.messages import SystemMessage, HumanMessage
    MODEL = os.environ.get("SLICE_MODEL", "claude-sonnet-4-6")
    API_KEY = os.environ.get("CLAUDE_KEY") or os.environ.get("ANTHROPIC_API_KEY")
    llm = ChatAnthropic(model=MODEL, temperature=0, max_tokens=1024, api_key=API_KEY)
    msgs = [SystemMessage(SYS[purpose]), HumanMessage(json.dumps(payload, ensure_ascii=False))]
    return llm.invoke(msgs).content

def _now(): return datetime.now(timezone.utc).isoformat(timespec="seconds")
def _log(state, actor, action, status, cause=None, rin="", rout=""):
    state["history"].append({"t":_now(),"actor":actor,"action":action,
        "input_digest":rin,"result_digest":rout,"status":status,"cause":cause})

## §1·결정2. TOOL 경계 = 실 MCP 서버
`mcp_model_tool.py`(별도 프로세스)를 stdio로 띄워 `web_search_specs` 도구를 호출한다. **이게 mock과의 결정적 차이** — 데이터가 내 코드가 아니라 외부 웹에서 온다.
- async라 주피터 셀에서 `await` 바로 가능.
- 반환은 MCP content 블록 리스트 → `json.loads(r[0]['text'])`로 파싱.

In [5]:
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

# command=sys.executable: 서버 서브프로세스를 커널과 '같은 파이썬'으로 띄움
#   (PATH의 python3가 커널과 다르면 서버에서 mcp/ddgs ModuleNotFoundError 방지).
# env=dict(os.environ): PATH·MCP_OFFLINE 상속. 맥에선 MCP_OFFLINE 없음 → 실 검색.
mcp_client = MultiServerMCPClient({
    "modeltool": {"command": sys.executable, "args": ["mcp_model_tool.py"],
                  "transport": "stdio", "env": dict(os.environ)}
})
MCP_TOOLS = await mcp_client.get_tools()
TOOLS_BY = {t.name: t for t in MCP_TOOLS}
print("MCP tools:", list(TOOLS_BY))

async def mcp_search(model: str) -> dict:
    raw = await TOOLS_BY["web_search_specs"].ainvoke({"model": model})
    return json.loads(raw[0]["text"]) if isinstance(raw, list) else raw

MCP tools: ['web_search_specs']


## §3~§4. 노드 5개
- **OFFER**: 현재 step의 행동 결정. 허용도구 검사(실행경로통제 1차).
- **TOOL**(async): s2에서 대상별로 MCP 웹검색(fan-out). 의도적 실패는 한 대상을 빈 결과로.
- **Verify**: s1~s3 결정적 구조검증 / s4 LLM-judge. **s2 = 대상별 스니펫·출처 존재**(LLM 안 씀).

In [6]:
STEP_ORDER = ["s1","s2","s3","s4"]

def plan_node(state):
    if not state["plan"]:
        crit = state["constraints"]["definition"]["criteria"]
        plan = [
            {"step_id":"s1","subgoal":"비교기준 정의","priority":1,"depends_on":[],"status":"pending"},
            {"step_id":"s2","subgoal":"대상별 웹검색(fan-out)","priority":2,"depends_on":["s1"],"status":"pending"},
            {"step_id":"s3","subgoal":"차이분석","priority":3,"depends_on":["s2"],"status":"pending"},
            {"step_id":"s4","subgoal":"결과요약","priority":4,"depends_on":["s3"],"status":"pending"},
        ]
        _log(state,"PLAN","build_plan","ok",rout=f"{len(plan)} steps, criteria={crit}")
        return {"plan":plan,"cursor":"s1"}
    cur = state["cursor"]
    nxt = next((s["step_id"] for s in state["plan"] if s["status"]=="pending"), None)
    _log(state,"PLAN","advance","ok",rin=f"from {cur}",rout=f"to {nxt}")
    return {"cursor":nxt}

def offer_node(state):
    cur = state["cursor"]; targets = state["constraints"]["definition"]["targets"]
    scratch = dict(state["scratch"])
    if cur == "s1":
        scratch["action"] = {"type":"define","criteria":state["constraints"]["definition"]["criteria"]}
    elif cur == "s2":
        if "web_search_specs" not in state["constraints"]["allowed_tools"]:
            _log(state,"OFFER","reject_tool","fail",cause="tool_not_allowed")
            return {"scratch":scratch,"verdict":{"passed":False,"cause":"tool_not_allowed"}}
        retrying = scratch.get("s2_retry", False)
        force = os.environ.get("FORCE_S2_FAIL") == "1"   # 차단기 데모: 항상 실패
        broken = force or not retrying                    # 첫 시도 한 대상 검색실패(+force면 영구)
        scratch["action"] = {"type":"extract","targets":targets,
                             "fail_target":(targets[-1] if broken else None)}
    elif cur == "s3": scratch["action"] = {"type":"compare"}
    elif cur == "s4": scratch["action"] = {"type":"summarize"}
    _log(state,"OFFER",f"decide:{scratch['action']['type']}","ok",rin=cur)
    return {"scratch":scratch}

async def tool_node(state):
    act = state["scratch"].get("action", {}); artifacts = dict(state["artifacts"]); t = act.get("type")
    if t == "define":
        artifacts["criteria"] = act["criteria"]; _log(state,"TOOL","define","ok",rout=str(act["criteria"]))
    elif t == "extract":
        specs = {}
        for m in act["targets"]:
            if act.get("fail_target") == m:                       # 의도적 실패: 빈 결과
                specs[m] = {"model":m,"snippets":[],"sources":[]}
            else:
                specs[m] = await mcp_search(m)                    # 실 MCP 웹검색
        artifacts["specs"] = specs
        got = sum(1 for s in specs.values() if s.get("sources"))
        _log(state,"TOOL","web_search","ok",rout=f"{got}/{len(specs)} targets via MCP")
    elif t == "compare":
        artifacts["analysis"] = llm_call("compare", {"criteria":state["constraints"]["definition"]["criteria"],
                                                     "specs":state["artifacts"].get("specs",{})}); _log(state,"TOOL","compare","ok")
    elif t == "summarize":
        artifacts["summary"] = llm_call("summarize", {"analysis":state["artifacts"].get("analysis",""),
                                                      "specs":state["artifacts"].get("specs",{})}); _log(state,"TOOL","summarize","ok")
    return {"artifacts":artifacts}

def memory_node(state):
    b = dict(state["budget"]); b["iter"] += 1
    _log(state,"MEMORY","checkpoint","ok",rout=f"iter={b['iter']}")
    return {"budget":b}

def verify_node(state):
    cur = state["cursor"]; crit = state["constraints"]["definition"]["criteria"]
    targets = state["constraints"]["definition"]["targets"]; art = state["artifacts"]
    if cur == "s1":
        v = {"passed":bool(art.get("criteria")),"cause":None if art.get("criteria") else "empty_criteria"}
    elif cur == "s2":
        specs = art.get("specs",{})
        ok = all(specs.get(m,{}).get("snippets") and specs.get(m,{}).get("sources") for m in targets)
        v = {"passed":ok,"cause":None if ok else "missing_source"}
    elif cur == "s3":
        v = {"passed":bool(art.get("analysis")),"cause":None if art.get("analysis") else "no_analysis"}
    elif cur == "s4":
        sources = [s for sp in art.get("specs",{}).values() for s in sp.get("sources",[])]
        judge = llm_call("judge", {"summary":art.get("summary",""),"criteria":crit,"sources":sources})
        ok = judge.strip().upper().startswith("PASS")
        v = {"passed":ok,"cause":None if ok else "judge_reject","detail":judge[:300]}
    else:
        v = {"passed":False,"cause":"unknown_step"}
    _log(state,"Verify",f"verify:{cur}","ok" if v["passed"] else "fail",cause=v["cause"])
    return {"verdict":v}

## §5·§7. DIAGNOSIS + ORC(순수 라우터)
ORC는 LLM을 안 쓴다. 모든 예산·verdict 기록을 ORC가 소유하고 결정을 `scratch["_route"]`에 적재 → `route()`는 읽기만.

In [7]:
DIAGNOSIS = {                       # 원인코드 -> (retriable?, corrective)
    "missing_source":    (True, "s2_retry"),
    "no_analysis":       (True, None),
    "judge_reject":      (True, None),
    "tool_not_allowed":  (False, None),
}
def _retriable(v): return DIAGNOSIS.get(v.get("cause"), (False,None))[0]
def _is_last_step(state): return state["cursor"] == STEP_ORDER[-1]

def orc_node(state):
    v = state["verdict"]; b = dict(state["budget"]); scratch = dict(state["scratch"])
    if v is None:
        scratch["_route"] = "PLAN"; return {"scratch":scratch}
    if v["passed"]:
        b["consec_fail"] = 0
        plan = [dict(s) for s in state["plan"]]
        for s in plan:
            if s["step_id"] == state["cursor"]: s["status"] = "done"
        scratch["_route"] = "END" if _is_last_step(state) else "PLAN"
        _log(state,"ORC","route:pass","ok",rin=str(state["cursor"]),rout=scratch["_route"])
        return {"budget":b,"verdict":None,"scratch":scratch,"plan":plan}
    if b["consec_fail"] >= b["fail_threshold"] or b["retry"] <= 0:
        scratch["_route"] = "FAIL"; scratch["_status"] = "FAILED"
        _log(state,"ORC","route:halt","fail",cause=v["cause"],rout="FAIL")
        return {"budget":b,"scratch":scratch}
    b["retry"] -= 1; b["consec_fail"] += 1
    retriable, corrective = DIAGNOSIS.get(v["cause"], (False,None))
    if retriable:
        if corrective: scratch[corrective] = True
        scratch["_route"] = "OFFER"
    else:
        scratch["_route"] = "PLAN"
    _log(state,"ORC","route:retry","ok",rin=f"cause={v['cause']}",
         rout=f"{scratch['_route']} retry={b['retry']} consec={b['consec_fail']}")
    return {"budget":b,"verdict":None,"scratch":scratch}

def route(state): return state["scratch"]["_route"]

## 그래프 조립 + checkpointer/Store
`tool_node`가 async라도 `add_node`에 그대로 등록 → `app.ainvoke`로 실행.

In [8]:
import aiosqlite
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver   # async invoke와 짝 (결정3: SqliteSaver의 async판)
from langgraph.store.memory import InMemoryStore

_aconn = await aiosqlite.connect(":memory:")     # 단기 메모리(checkpoint) 백엔드
CHECKPOINTER = AsyncSqliteSaver(_aconn)
STORE = InMemoryStore()                          # 장기 메모리

def build_app():
    g = StateGraph(AgentContext)
    for name, fn in [("ORC",orc_node),("PLAN",plan_node),("OFFER",offer_node),
                     ("TOOL",tool_node),("MEMORY",memory_node),("VERIFY",verify_node)]:
        g.add_node(name, fn)
    g.add_edge(START, "ORC")
    g.add_conditional_edges("ORC", route, {"PLAN":"PLAN","OFFER":"OFFER","END":END,"FAIL":END})
    g.add_edge("PLAN","OFFER"); g.add_edge("OFFER","TOOL")
    g.add_edge("TOOL","MEMORY"); g.add_edge("MEMORY","VERIFY"); g.add_edge("VERIFY","ORC")
    return g.compile(checkpointer=CHECKPOINTER, store=STORE)

def initial_state():
    return {"goal":"건설자재 객체탐지 모델 후보 3종(Faster R-CNN/YOLOv8/RT-DETR)을 비교하라",
        "constraints":{"definition":{"criteria":["mAP","FPS","train_cost","deploy_difficulty"],
                       "targets":["Faster R-CNN","YOLOv8","RT-DETR"]},
            "allowed_tools":["web_search_specs"],"forbidden":["rm -rf","DROP TABLE"],
            "acceptance":{"final":["all_criteria_covered","sources_present"]}},
        "plan":[],"cursor":None,"artifacts":{},"scratch":{},"history":[],"verdict":None,
        "budget":{"iter":0,"max_iter":12,"retry":3,"consec_fail":0,"fail_threshold":3}}

async def run(tag, thread_id):
    app = build_app()
    cfg = {"recursion_limit":100,"configurable":{"thread_id":thread_id}}
    final = await app.ainvoke(initial_state(), config=cfg)
    print(f"\n===== {tag} =====")
    for h in final["history"]:
        cause = f" cause={h['cause']}" if h["cause"] else ""
        extra = f" :: {h['result_digest']}" if h["result_digest"] else ""
        print(f"  [{h['actor']:<6}] {h['action']:<16} {h['status']}{cause}{extra}")
    status = final["scratch"].get("_status","DONE")
    print(f"  --- status={status} iter={final['budget']['iter']} "
          f"retry_left={final['budget']['retry']} consec_fail={final['budget']['consec_fail']}")
    # 성공/실패 무관하게 가져온 데이터·출처를 보여준다
    specs = final["artifacts"].get("specs", {})
    if specs:
        print("  --- 출처(real URL):")
        for m, sp in specs.items():
            print(f"        {m}: {sp.get('sources', [])[:2]}")
    if final["artifacts"].get("summary"):
        print(f"  --- summary: {final['artifacts']['summary']}")
    v = final.get("verdict")
    if status == "FAILED" and v and v.get("detail"):
        print(f"  --- 실패사유(judge): {v['detail']}")
    print("  --- plan:", {s['step_id']:s['status'] for s in final['plan']})
    return final

## 시나리오 A — happy path
s2 첫 검색에서 한 대상을 빈 결과로(검색 실패 시뮬) → Verify가 `missing_source` 잡음 → 재시도 회복 → s4까지 완주. 출처가 **실제 URL**로 찍힌다.

In [9]:
a = await run("A. happy path", "slice-A")


===== A. happy path =====
  [PLAN  ] build_plan       ok :: 4 steps, criteria=['mAP', 'FPS', 'train_cost', 'deploy_difficulty']
  [OFFER ] decide:define    ok
  [TOOL  ] define           ok :: ['mAP', 'FPS', 'train_cost', 'deploy_difficulty']
  [MEMORY] checkpoint       ok :: iter=1
  [Verify] verify:s1        ok
  [ORC   ] route:pass       ok :: PLAN
  [PLAN  ] advance          ok :: to s2
  [OFFER ] decide:extract   ok
  [TOOL  ] web_search       ok :: 2/3 targets via MCP
  [MEMORY] checkpoint       ok :: iter=2
  [Verify] verify:s2        fail cause=missing_source
  [ORC   ] route:retry      ok :: OFFER retry=2 consec=1
  [OFFER ] decide:extract   ok
  [TOOL  ] web_search       ok :: 3/3 targets via MCP
  [MEMORY] checkpoint       ok :: iter=3
  [Verify] verify:s2        ok
  [ORC   ] route:pass       ok :: PLAN
  [PLAN  ] advance          ok :: to s3
  [OFFER ] decide:compare   ok
  [TOOL  ] compare          ok
  [MEMORY] checkpoint       ok :: iter=4
  [Verify] verify:s3        o

## 시나리오 B — 차단기
`FORCE_S2_FAIL=1`로 s2가 매번 실패. 유계 재시도(3회) 소진 후 ORC가 `FAIL`로 정지 — 무한루프 없음.

In [10]:
os.environ["FORCE_S2_FAIL"] = "1"
b = await run("B. circuit breaker", "slice-B")
os.environ.pop("FORCE_S2_FAIL", None)


===== B. circuit breaker =====
  [PLAN  ] build_plan       ok :: 4 steps, criteria=['mAP', 'FPS', 'train_cost', 'deploy_difficulty']
  [OFFER ] decide:define    ok
  [TOOL  ] define           ok :: ['mAP', 'FPS', 'train_cost', 'deploy_difficulty']
  [MEMORY] checkpoint       ok :: iter=1
  [Verify] verify:s1        ok
  [ORC   ] route:pass       ok :: PLAN
  [PLAN  ] advance          ok :: to s2
  [OFFER ] decide:extract   ok
  [TOOL  ] web_search       ok :: 2/3 targets via MCP
  [MEMORY] checkpoint       ok :: iter=2
  [Verify] verify:s2        fail cause=missing_source
  [ORC   ] route:retry      ok :: OFFER retry=2 consec=1
  [OFFER ] decide:extract   ok
  [TOOL  ] web_search       ok :: 2/3 targets via MCP
  [MEMORY] checkpoint       ok :: iter=3
  [Verify] verify:s2        fail cause=missing_source
  [ORC   ] route:retry      ok :: OFFER retry=1 consec=2
  [OFFER ] decide:extract   ok
  [TOOL  ] web_search       ok :: 2/3 targets via MCP
  [MEMORY] checkpoint       ok :: iter=4


'1'

## 위조 테스트 — "짜고 친 거 아닌가" 검증
mock일 땐 데이터가 내 코드라 의심이 남았다. 이제 데이터는 MCP 웹검색에서 온다. 확인법: 시나리오 A의 `출처(real URL)`가 **실제 접속 가능한 외부 URL**인지 직접 클릭해 보라. 또한 `mcp_model_tool.py`의 검색 쿼리를 바꾸면(예: 다른 모델명) 결론이 따라 바뀐다 — 내 기억이 아니라 외부 데이터를 따른다는 증거.

---
## 다음 단계
1. **결과 검토:** A의 analysis/summary와 출처 URL을 보고 품질 판단.
2. **.py 모듈 졸업:** 노드별 `agents/*.py`로 분리 + git (Cursor). async MCP는 `asyncio.run` 래퍼.
3. **유료 검색 승급(선택):** 키리스 DDG → Tavily 등으로 정확도↑.